In [ ]:
n_samples = 10800 * 60 * 10  # 6,480,000 samples

In [ ]:
from pathlib import Path
import h5py


def inspect_h5(path):
    path = Path(path)

    print(f"\nFile: {path}")
    print(f"Size on disk: {path.stat().st_size / 1024**3:.3f} GB")

    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(
                    f"{name}: "
                    f"shape={obj.shape}, "
                    f"dtype={obj.dtype}, "
                    f"chunks={obj.chunks}, "
                    f"compression={obj.compression}, "
                    f"storage={obj.id.get_storage_size() / 1024**3:.3f} GB"
                )

        f.visititems(visitor)


inspect_h5(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\analysis\derived\voltage\voltage_session_traces_dff_robust_f0_trial.h5")
inspect_h5(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\analysis\derived\voltage\robust_f0_FIRST_10_MIN.h5")


In [ ]:
from pathlib import Path
import h5py
import numpy as np


def copy_attrs(src, dst):
    for key, value in src.attrs.items():
        dst.attrs[key] = value


def creation_kwargs_like_source(src_dset, out_shape):
    kwargs = {"dtype": src_dset.dtype}

    if src_dset.chunks is not None:
        chunks = tuple(min(c, s) for c, s in zip(src_dset.chunks, out_shape))
        kwargs["chunks"] = chunks

    if src_dset.compression is not None:
        kwargs["compression"] = src_dset.compression
        kwargs["compression_opts"] = src_dset.compression_opts

    if src_dset.shuffle:
        kwargs["shuffle"] = True

    if src_dset.fletcher32:
        kwargs["fletcher32"] = True

    if src_dset.scaleoffset is not None:
        kwargs["scaleoffset"] = src_dset.scaleoffset

    return kwargs


def copy_dataset_truncated_last_axis(
    src_dset,
    dst_group,
    name,
    keep_n,
    block_samples=250_000,
):
    """
    For datasets shaped roi x sample, keep src[:, :keep_n].
    """

    out_shape = src_dset.shape[:-1] + (min(keep_n, src_dset.shape[-1]),)
    kwargs = creation_kwargs_like_source(src_dset, out_shape)

    dst_dset = dst_group.create_dataset(name, shape=out_shape, **kwargs)
    copy_attrs(src_dset, dst_dset)

    n_keep = out_shape[-1]

    for start in range(0, n_keep, block_samples):
        stop = min(start + block_samples, n_keep)
        dst_dset[..., start:stop] = src_dset[..., start:stop]

    dst_dset.attrs["truncated"] = True
    dst_dset.attrs["truncated_axis"] = -1
    dst_dset.attrs["kept_samples"] = n_keep

    return dst_dset


def copy_1d_truncated(
    src_dset,
    dst_group,
    name,
    keep_n,
):
    """
    For 1D datasets shaped sample, keep src[:keep_n].
    """

    n_keep = min(keep_n, src_dset.shape[0])
    out_shape = (n_keep,)
    kwargs = creation_kwargs_like_source(src_dset, out_shape)

    dst_dset = dst_group.create_dataset(name, shape=out_shape, **kwargs)
    copy_attrs(src_dset, dst_dset)

    dst_dset[:] = src_dset[:n_keep]

    dst_dset.attrs["truncated"] = True
    dst_dset.attrs["truncated_axis"] = 0
    dst_dset.attrs["kept_samples"] = n_keep

    return dst_dset


def truncate_voltage_file_for_upload(
    input_path,
    output_path,
    keep_seconds=600,
    sample_rate_hz=10800,
    keep_signals=("dff",),
    dmd_groups=("DMD1", "DMD2"),
):
    """
    Make a smaller upload-friendly voltage .h5 file.

    Expected structure:
        DMD1/dff      shape = roi x sample
        DMD1/raw_f    shape = roi x sample
        DMD1/f0       shape = roi x sample
        DMD1/timebase_sec shape = sample

    Parameters
    ----------
    keep_signals:
        Which signal arrays to keep and truncate.
        Examples:
            ("dff",)
            ("raw_f",)
            ("dff", "f0", "raw_f")
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    if output_path.exists():
        raise FileExistsError(f"Output file already exists: {output_path}")

    keep_n = int(round(keep_seconds * sample_rate_hz))
    keep_signals = set(keep_signals)
    dmd_groups = set(dmd_groups)

    n_truncated = 0
    n_skipped_signals = 0

    with h5py.File(input_path, "r") as src, h5py.File(output_path, "w") as dst:
        copy_attrs(src, dst)

        for group_name, src_group in src.items():
            if not isinstance(src_group, h5py.Group):
                src.copy(src_group, dst, name=group_name)
                continue

            dst_group = dst.create_group(group_name)
            copy_attrs(src_group, dst_group)

            for name, item in src_group.items():
                item_path = f"{group_name}/{name}"

                if not isinstance(item, h5py.Dataset):
                    src_group.copy(item, dst_group, name=name)
                    continue

                is_dmd_group = group_name in dmd_groups
                is_signal = name in {"dff", "raw_f", "f0"}
                should_keep_signal = name in keep_signals

                if is_dmd_group and is_signal and should_keep_signal:
                    print(
                        f"Truncating {item_path}: "
                        f"{item.shape} -> {item.shape[:-1] + (min(keep_n, item.shape[-1]),)}"
                    )

                    copy_dataset_truncated_last_axis(
                        src_dset=item,
                        dst_group=dst_group,
                        name=name,
                        keep_n=keep_n,
                    )
                    n_truncated += 1

                elif is_dmd_group and is_signal and not should_keep_signal:
                    print(f"Skipping signal dataset {item_path}: {item.shape}")
                    n_skipped_signals += 1

                elif is_dmd_group and name == "timebase_sec":
                    print(
                        f"Truncating {item_path}: "
                        f"{item.shape} -> {(min(keep_n, item.shape[0]),)}"
                    )

                    copy_1d_truncated(
                        src_dset=item,
                        dst_group=dst_group,
                        name=name,
                        keep_n=keep_n,
                    )
                    n_truncated += 1

                else:
                    print(f"Copying unchanged {item_path}: {item.shape}")
                    src_group.copy(item, dst_group, name=name)

    if n_truncated == 0:
        raise RuntimeError(
            "No datasets were truncated. Check dataset names and group names."
        )

    original_gb = input_path.stat().st_size / 1024**3
    output_gb = output_path.stat().st_size / 1024**3

    print("\nDone.")
    print(f"Original size:  {original_gb:.3f} GB")
    print(f"Output size:    {output_gb:.3f} GB")
    print(f"Truncated datasets: {n_truncated}")
    print(f"Skipped signal datasets: {n_skipped_signals}")
    print(f"Saved to: {output_path}")

In [ ]:
truncate_voltage_file_for_upload(
    input_path=r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\analysis\derived\voltage\voltage_session_traces_dff_robust_f0_trial.h5",
    output_path=r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\analysis\derived\voltage\voltage_session_traces_dff_FIRST_5_MIN.h5",
    keep_seconds=300,
    sample_rate_hz=10800,
    keep_signals=("dff",),
)

In [ ]:
from pathlib import Path
import shutil
import json
import h5py
import numpy as np


# =============================================================================
# User settings
# =============================================================================

SUMMARY_PATH = Path(
    r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\826031_2026-01-30_15-04-02_slap2_2026-01-30_15-04-02\source_extraction\dendriticVoltageExtraction\dendriticVoltageSummary-260603-140324.mat")

# Set this to None to auto-detect from the summary folder.
# Or set it explicitly if auto-detection fails.
TRACE_H5_PATH = None
# TRACE_H5_PATH = Path(
#     r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\tmp\826031_2026-01-30_15-04-02\analysis\derived\voltage\dendriticVoltageTraces-260130-150402.h5"
# )

OUT_DIR = SUMMARY_PATH.parent / "first_10_min_example"

KEEP_MINUTES = 2
LINE_RATE_HZ = 10_800

# If True, copied summary .mat is left mostly unchanged, with a JSON sidecar
# documenting the new paired H5. This is safest for MATLAB v7.3 structs.
COPY_SUMMARY_MAT = True


# =============================================================================
# Helper functions
# =============================================================================

def file_size_gb(path: Path) -> float:
    return path.stat().st_size / 1024**3


def find_trace_h5(summary_path: Path) -> Path:
    """
    Try to find the paired dendriticVoltageTraces*.h5 file next to the summary.
    """
    candidates = sorted(summary_path.parent.glob("dendriticVoltageTraces*.h5"))

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not auto-detect paired trace H5 in:\n{summary_path.parent}\n\n"
            "Set TRACE_H5_PATH explicitly."
        )

    raise RuntimeError(
        "Multiple candidate trace H5 files found:\n"
        + "\n".join(str(p) for p in candidates)
        + "\n\nSet TRACE_H5_PATH explicitly."
    )


def copy_attrs(src_obj, dst_obj):
    for key, value in src_obj.attrs.items():
        dst_obj.attrs[key] = value


def create_group_recursive(dst_file, group_path):
    """
    Ensure group_path exists in dst_file.
    """
    if group_path in ("", "/"):
        return dst_file["/"]

    parts = [p for p in group_path.strip("/").split("/") if p]
    current = dst_file["/"]

    for part in parts:
        if part not in current:
            current = current.create_group(part)
        else:
            current = current[part]

    return current


def dataset_creation_kwargs_like_source(src_dset, out_shape):
    """
    Preserve chunking/compression when possible.
    """
    kwargs = {
        "dtype": src_dset.dtype,
    }

    if src_dset.chunks is not None:
        chunks = tuple(max(1, min(c, s)) for c, s in zip(src_dset.chunks, out_shape))
        kwargs["chunks"] = chunks

    if src_dset.compression is not None:
        kwargs["compression"] = src_dset.compression
        kwargs["compression_opts"] = src_dset.compression_opts

    if src_dset.shuffle:
        kwargs["shuffle"] = True

    if src_dset.fletcher32:
        kwargs["fletcher32"] = True

    if src_dset.scaleoffset is not None:
        kwargs["scaleoffset"] = src_dset.scaleoffset

    return kwargs


def is_trace_dataset(path, dset):
    """
    Identify trace datasets generated by extractDendrites_new.m.

    Handles likely forms:
        /traces/trial_0001
        /traces/trial_0002
        /traces/continuous/DMD1
        /traces/continuous/DMD2
    """
    clean = path.strip("/")

    if clean.startswith("traces/trial_"):
        return True

    if clean.startswith("traces/continuous/"):
        return True

    return False


def infer_time_axis(shape):
    """
    For trace data, the time/line dimension should be much larger than ROI count.
    This is robust to MATLAB/Python dimension-order differences.
    """
    if len(shape) == 0:
        return None

    if len(shape) == 1:
        return 0

    return int(np.argmax(shape))


def copy_dataset_full(src_dset, dst_group, name):
    """
    Copy an entire dataset.
    """
    src_dset.parent.copy(src_dset, dst_group, name=name)


def copy_dataset_truncated_along_axis(
    src_dset,
    dst_group,
    name,
    keep_n,
    axis,
    block_samples=250_000,
):
    """
    Copy a dataset while truncating a selected axis.

    For example:
        roi x sample   with axis=1
        sample x roi   with axis=0
    """
    out_shape = list(src_dset.shape)
    out_shape[axis] = min(keep_n, src_dset.shape[axis])
    out_shape = tuple(out_shape)

    kwargs = dataset_creation_kwargs_like_source(src_dset, out_shape)

    dst_dset = dst_group.create_dataset(
        name,
        shape=out_shape,
        **kwargs,
    )
    copy_attrs(src_dset, dst_dset)

    n_keep = out_shape[axis]

    for start in range(0, n_keep, block_samples):
        stop = min(start + block_samples, n_keep)

        sl = [slice(None)] * src_dset.ndim
        sl[axis] = slice(start, stop)
        sl = tuple(sl)

        dst_dset[sl] = src_dset[sl]

    dst_dset.attrs["truncated"] = True
    dst_dset.attrs["truncated_axis"] = axis
    dst_dset.attrs["kept_samples"] = n_keep

    return dst_dset


def copy_and_truncate_extract_dendrites_h5(
    input_h5,
    output_h5,
    keep_minutes,
    line_rate_hz,
):
    """
    Copy an extractDendrites_new.m trace H5 while retaining only the first
    keep_minutes worth of trace rows/samples.

    For trial-mode output, this accumulates across:
        /traces/trial_0001
        /traces/trial_0002
        ...

    For continuous-mode output, this keeps the first X minutes from each:
        /traces/continuous/DMD1
        /traces/continuous/DMD2
    """
    input_h5 = Path(input_h5)
    output_h5 = Path(output_h5)

    keep_n = int(round(keep_minutes * 60 * line_rate_hz))

    if output_h5.exists():
        raise FileExistsError(f"Output H5 already exists:\n{output_h5}")

    truncated_paths = []
    skipped_trial_paths = []

    # For trial-mode output, this tracks cumulative rows across trials.
    trial_remaining = keep_n

    with h5py.File(input_h5, "r") as src, h5py.File(output_h5, "w") as dst:
        copy_attrs(src, dst)

        def recurse(src_group, dst_group):
            nonlocal trial_remaining

            copy_attrs(src_group, dst_group)

            for name, item in src_group.items():
                item_path = item.name.strip("/")

                if isinstance(item, h5py.Group):
                    new_group = dst_group.create_group(name)
                    recurse(item, new_group)

                elif isinstance(item, h5py.Dataset):
                    if is_trace_dataset(item_path, item):
                        axis = infer_time_axis(item.shape)

                        if axis is None:
                            copy_dataset_full(item, dst_group, name)
                            continue

                        is_trial = item_path.startswith("traces/trial_")
                        is_continuous = item_path.startswith("traces/continuous/")

                        if is_trial:
                            if trial_remaining <= 0:
                                print(f"Skipping trial after cutoff: {item_path} {item.shape}")
                                skipped_trial_paths.append(item_path)
                                continue

                            keep_this = min(trial_remaining, item.shape[axis])
                            out_shape = list(item.shape)
                            out_shape[axis] = keep_this
                            out_shape = tuple(out_shape)

                            print(f"Truncating/copying trial {item_path}: {item.shape} -> {out_shape}")

                            copy_dataset_truncated_along_axis(
                                src_dset=item,
                                dst_group=dst_group,
                                name=name,
                                keep_n=keep_this,
                                axis=axis,
                            )

                            trial_remaining -= keep_this
                            truncated_paths.append(item_path)

                        elif is_continuous:
                            keep_this = min(keep_n, item.shape[axis])
                            out_shape = list(item.shape)
                            out_shape[axis] = keep_this
                            out_shape = tuple(out_shape)

                            print(f"Truncating continuous {item_path}: {item.shape} -> {out_shape}")

                            copy_dataset_truncated_along_axis(
                                src_dset=item,
                                dst_group=dst_group,
                                name=name,
                                keep_n=keep_this,
                                axis=axis,
                            )

                            truncated_paths.append(item_path)

                    else:
                        print(f"Copying unchanged {item_path}: {item.shape}")
                        copy_dataset_full(item, dst_group, name)

        recurse(src, dst)

    if len(truncated_paths) == 0:
        raise RuntimeError(
            "No trace datasets were truncated. Expected datasets like "
            "'/traces/trial_0001' or '/traces/continuous/DMD1'. "
            "Run h5py inspection on the file to check its structure."
        )

    return {
        "input_h5": str(input_h5),
        "output_h5": str(output_h5),
        "keep_minutes": keep_minutes,
        "line_rate_hz": line_rate_hz,
        "keep_samples": keep_n,
        "truncated_paths": truncated_paths,
        "skipped_trial_paths": skipped_trial_paths,
        "input_h5_size_gb": file_size_gb(input_h5),
        "output_h5_size_gb": file_size_gb(output_h5),
    }


def inspect_h5_brief(path):
    path = Path(path)
    print(f"\nFile: {path}")
    print(f"Size on disk: {file_size_gb(path):.3f} GB")

    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                storage = obj.id.get_storage_size() / 1024**3
                print(
                    f"{name}: shape={obj.shape}, dtype={obj.dtype}, "
                    f"chunks={obj.chunks}, compression={obj.compression}, "
                    f"storage={storage:.3f} GB"
                )

        f.visititems(visitor)


# =============================================================================
# Run
# =============================================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)

if TRACE_H5_PATH is None:
    TRACE_H5_PATH = find_trace_h5(SUMMARY_PATH)
else:
    TRACE_H5_PATH = Path(TRACE_H5_PATH)

summary_out = OUT_DIR / f"{SUMMARY_PATH.stem}_FIRST_{KEEP_MINUTES}_MIN{SUMMARY_PATH.suffix}"
h5_out = OUT_DIR / f"{TRACE_H5_PATH.stem}_FIRST_{KEEP_MINUTES}_MIN{TRACE_H5_PATH.suffix}"
sidecar_json = OUT_DIR / f"{SUMMARY_PATH.stem}_FIRST_{KEEP_MINUTES}_MIN_truncation_metadata.json"

print("Summary path:")
print(SUMMARY_PATH)
print("\nTrace H5 path:")
print(TRACE_H5_PATH)
print("\nOutput dir:")
print(OUT_DIR)

result = copy_and_truncate_extract_dendrites_h5(
    input_h5=TRACE_H5_PATH,
    output_h5=h5_out,
    keep_minutes=KEEP_MINUTES,
    line_rate_hz=LINE_RATE_HZ,
)

if COPY_SUMMARY_MAT:
    if summary_out.exists():
        raise FileExistsError(f"Output summary already exists:\n{summary_out}")

    shutil.copy2(SUMMARY_PATH, summary_out)

    result["input_summary"] = str(SUMMARY_PATH)
    result["output_summary"] = str(summary_out)
    result["note"] = (
        "Summary .mat was copied unchanged. Use the paired truncated H5 listed "
        "in output_h5. If downstream MATLAB code depends on summary.outputH5, "
        "manually update that field or use the JSON sidecar."
    )

with open(sidecar_json, "w") as f:
    json.dump(result, f, indent=2)

print("\nDone.")
print(f"Original H5 size: {result['input_h5_size_gb']:.3f} GB")
print(f"Output H5 size:   {result['output_h5_size_gb']:.3f} GB")
print(f"Output H5:        {h5_out}")
print(f"Output summary:   {summary_out}")
print(f"Sidecar JSON:     {sidecar_json}")

print("\nInspecting truncated H5:")
inspect_h5_brief(h5_out)